In [0]:

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

from pyspark.sql.functions import *

sales_df = spark.read.table(f"{catalog}.{schema}.sales_silver")
# sales_df.display()

gold_sales_mart = (
    sales_df
    .filter(col("order_status") == "COMPLETED")
    .withColumn(
        "order_year",
        year(to_date(col("signup_date"), "dd-MM-yyyy"))
    )
    .withColumn(
        "order_month",
        month(to_date(col("signup_date"), "dd-MM-yyyy"))
    )
    .withColumn(
        "order_month_name",
        date_format(to_date(col("signup_date"), "dd-MM-yyyy"), "MMMM")
    )
    .withColumn(
        "unit_price",
        round(col("amount") / col("quantity"), 2)
    )
    .withColumn(
        "customer_tenure_days",
         datediff(
        to_date(col("order_date"), "dd-MM-yyyy"),
        to_date(col("signup_date"), "dd-MM-yyyy")
    )
        )
    
)
# display(gold_sales_mart)

gold_sales_mart = gold_sales_mart.select(
    "order_id",
    "order_date",
    "order_year",
    "order_month",
    "order_month_name",

    "customer_id",
    "customer_name",
    "city",
    "state",
    "signup_date",
    "customer_tenure_days",

    "product_id",
    "product_name",
    "category",
    "brand",

    "payment_mode",

    "quantity",
    "unit_price",
    "amount"
)

(
    gold_sales_mart
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema}.gold_sales_mart")
)

# display(
#     gold_sales_mart)